# Hydrogen figure pipeline

Run this master notebook with the **R kernel**. It executes the eight component notebooks in dependency order, each in a separate R kernel, using one common project directory. Execution stops if a component fails or a required output is missing.

The primary progression, failure and delay analyses use discrete-time complementary log-log models with project-clustered inference. Supplementary Figures S3–S5 use a separate **discrete-time multinomial competing-risk model** with the mutually exclusive annual outcomes `No transition`, `Progress` and `Failure`. The multinomial probabilities sum to one by construction and are propagated recursively to obtain the cumulative incidence of progression in the presence of failure.

The governance/institutional control is selected once in the configuration cell and is used consistently in the country/macro specifications of the progression, failure, delay and multinomial competing-risk models. Choose `Governance_Score`, `Ease_of_doing_business`, or `none`. Both candidate variables are prepared and standardised upstream.

Across figures and coefficient tables, the project-duration covariate `prev_time_in_status` is labelled **Time in stage**; the internal variable name is retained in code and model objects.

All figure notebooks use enlarged publication typography, with text sizes set 50\% above the previous figure specification. Figure canvases and margins provide additional room for the larger labels while retaining the same analytical content.

## Main figure index

| Figure | Content | Component notebook | PDF output |
|---|---|---|---|
| 1 | Project development flows, durations and composition | Figure-Sankey | `figure_sankey.pdf` |
| 2 | Project progression and failure coefficients | Figure-Tornado | `figure_tornado.pdf` |
| 3 | Progression by end-use profile | Figure-Demand | `figure_demand.pdf` |
| 4 | Progression by project capacity | Figure-Capacity | `figure_capacity.pdf` |
| 5 | Country-risk progression and country composition | Figure-Regions | `figure_country_risk.pdf` |
| 6 | First schedule revisions / delays | Figure-Delays | `figure_delay.pdf` |

`figure_tornado_full.pdf` is the full coefficient-chart export associated with Figure 2.

## Supplementary figure index

| Figure | Content | Component notebook | PDF output |
|---|---|---|---|
| S1 | Global hydrogen project map | Figure-Sankey | `figure_S1_project_map.pdf` |
| S2 | Variable distribution | Figure-Tornado | `figure_S2_variable_distribution.pdf` |
| S3 | Competing risk model Demand | Figure-Demand | `figure_S3_competing_risk_demand.pdf` |
| S4 | Competing risk model Capacity | Figure-Capacity | `figure_S4_competing_risk_capacity.pdf` |
| S5 | Competing risk model Regions | Figure-Regions | `figure_S5_competing_risk_regions.pdf` |
| S6 | Capacity by end use | Figure-Sankey | `figure_S6_capacity_by_end_use.pdf` |
| S7 | Delays by stage | Figure-Delays | `figure_S7_delays_by_stage.pdf` |

## Setup

1. Place all notebooks in one folder and open this master notebook with the R kernel.
2. Set `notebook_dir` to the notebook folder and `project_dir` to the H2 project directory containing `Data/` and the other source inputs referenced by Data Preparation. The two paths may be identical.
3. Set `governance_control` to `"Governance_Score"`, `"Ease_of_doing_business"`, or `"none"`.
4. Run all cells. Keep `run_data_preparation = TRUE` for a complete rebuild. If it is `FALSE`, all prepared RDS inputs must already exist.

The count audit exports `sample_count_reconciliation.csv` plus dataset-specific summaries, yearly counts, duplicate-key details and flagged source rows. Figure-Sankey writes `Figure1_ST1_ST2_sample_count_caption_notes.txt` with run-specific disclosure text. Re-run Data Preparation before Figure-Sankey to generate the matching audit.

The source-data files and R packages are not bundled. Component notebooks overwrite their named outputs when rerun.


**Canonical package note:** the component notebook filenames in this package match the names expected by the master runner. The deprecated Sankey notebook is retained for provenance only and is not part of the execution pipeline.


## Configuration


In [ ]:
# Edit these paths if the notebook folder differs from the project/data folder.
notebook_dir <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
project_dir <- normalizePath(
  Sys.getenv("H2_PROJECT_DIR", unset = notebook_dir),
  winslash = "/", mustWork = TRUE
)

run_data_preparation <- TRUE
jupyter_command <- Sys.which("jupyter")

# Governance / institutional-quality control used in every country/macro model.
# Choose exactly one of:
#   "Governance_Score"
#   "Ease_of_doing_business"
#   "none"
governance_control <- "Governance_Score"

valid_governance_controls <- c(
  "Governance_Score",
  "Ease_of_doing_business",
  "none"
)

if (!governance_control %in% valid_governance_controls) {
  stop(
    paste0(
      "governance_control must be one of: ",
      paste(valid_governance_controls, collapse = ", ")
    )
  )
}

pipeline_notebooks <- c(
  "Data-Preparation.ipynb",
  "Figure-Tornado.ipynb",
  "Discrete-Time-Multinomial-Competing-Risk-Model.ipynb",
  "Figure-Demand.ipynb",
  "Figure-Capacity.ipynb",
  "Figure-Regions.ipynb",
  "Figure-Sankey.ipynb",
  "Figure-Delays.ipynb"
)

main_outputs <- c(
  Figure1 = "figure_sankey.pdf",
  Figure2 = "figure_tornado.pdf",
  Figure3 = "figure_demand.pdf",
  Figure4 = "figure_capacity.pdf",
  Figure5 = "figure_country_risk.pdf",
  Figure6 = "figure_delay.pdf"
)

supplementary_outputs <- c(
  S1 = "figure_S1_project_map.pdf",
  S2 = "figure_S2_variable_distribution.pdf",
  S3 = "figure_S3_competing_risk_demand.pdf",
  S4 = "figure_S4_competing_risk_capacity.pdf",
  S5 = "figure_S5_competing_risk_regions.pdf",
  S6 = "figure_S6_capacity_by_end_use.pdf",
  S7 = "figure_S7_delays_by_stage.pdf"
)

expected_outputs <- list(
  "Data-Preparation.ipynb" = c(
    "master_data.rds",
    "cloglog_data_full.rds",
    "cloglog_data_processed.rds",
    "sample_count_audit.rds",
    "sample_count_reconciliation.csv"
  ),
  "Figure-Tornado.ipynb" = c(
    "waterfall_results.rds",
    "figure_tornado.pdf",
    "figure_tornado_full.pdf",
    supplementary_outputs[["S2"]]
  ),
  "Discrete-Time-Multinomial-Competing-Risk-Model.ipynb" = c(
    "multinomial_competing_risk_results.rds",
    "multinomial_competing_risk_coefficients.csv",
    "multinomial_competing_risk_model_statistics.csv"
  ),
  "Figure-Demand.ipynb" = c(
    main_outputs[["Figure3"]],
    supplementary_outputs[["S3"]]
  ),
  "Figure-Capacity.ipynb" = c(
    main_outputs[["Figure4"]],
    supplementary_outputs[["S4"]]
  ),
  "Figure-Regions.ipynb" = c(
    main_outputs[["Figure5"]],
    supplementary_outputs[["S5"]]
  ),
  "Figure-Sankey.ipynb" = c(
    main_outputs[["Figure1"]],
    supplementary_outputs[["S1"]],
    supplementary_outputs[["S6"]],
    "Figure1_ST1_ST2_sample_count_caption_notes.txt",
    "sample_count_plotting_only_pairs.csv"
  ),
  "Figure-Delays.ipynb" = c(
    "delay_waterfall_results.rds",
    main_outputs[["Figure6"]],
    supplementary_outputs[["S7"]]
  )
)


## Preflight


In [ ]:
# Fail before a long run if a package used by the component notebooks is absent.
required_r_packages <- c(
  "dplyr", "tidyr", "purrr", "tibble", "stringr", "readxl", "cellranger",
  "sf", "terra", "units", "rnaturalearth", "sandwich", "lmtest", "pROC",
  "ggplot2", "ggalluvial", "ggrepel", "ggsci", "scales", "patchwork", "MASS", "geepack", "nnet"
)
missing_r_packages <- required_r_packages[
  !vapply(required_r_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(missing_r_packages) > 0L) {
  stop(paste("Install required R packages before running:",
             paste(missing_r_packages, collapse = ", ")))
}

if (!nzchar(jupyter_command)) {
  stop("jupyter is not on PATH. Install Jupyter/nbconvert in the environment running this R kernel.")
}
missing_notebooks <- pipeline_notebooks[
  !file.exists(file.path(notebook_dir, pipeline_notebooks))
]
if (length(missing_notebooks) > 0L) {
  stop(paste("Missing pipeline notebooks:", paste(missing_notebooks, collapse = ", ")))
}

if (!run_data_preparation) {
  pipeline_notebooks <- setdiff(pipeline_notebooks, "Data-Preparation.ipynb")
  prepared_inputs <- expected_outputs[["Data-Preparation.ipynb"]]
  missing_inputs <- prepared_inputs[!file.exists(file.path(project_dir, prepared_inputs))]
  if (length(missing_inputs) > 0L) {
    stop(paste("Missing prepared inputs:", paste(missing_inputs, collapse = ", ")))
  }
}

message("Notebook directory: ", notebook_dir)
message("Project/data directory: ", project_dir)
message("Governance control: ", governance_control)
message("Execution order:")
print(pipeline_notebooks)


## Execute the component notebooks


In [ ]:
check_fresh_outputs <- function(files, started_at) {
  paths <- file.path(project_dir, files)
  info <- file.info(paths)

  data.frame(
    file = files,
    exists = file.exists(paths),
    nonempty = !is.na(info$size) & info$size > 0,
    updated_this_run = !is.na(info$mtime) & info$mtime >= started_at - 2,
    stringsAsFactors = FALSE,
    row.names = NULL
  )
}

run_pipeline <- function() {
  previous_project_dir <- Sys.getenv("H2_PROJECT_DIR", unset = NA_character_)
  previous_governance_control <- Sys.getenv(
    "H2_GOVERNANCE_CONTROL",
    unset = NA_character_
  )

  on.exit(
    {
      if (is.na(previous_project_dir)) {
        Sys.unsetenv("H2_PROJECT_DIR")
      } else {
        Sys.setenv(H2_PROJECT_DIR = previous_project_dir)
      }

      if (is.na(previous_governance_control)) {
        Sys.unsetenv("H2_GOVERNANCE_CONTROL")
      } else {
        Sys.setenv(
          H2_GOVERNANCE_CONTROL = previous_governance_control
        )
      }
    },
    add = TRUE
  )

  Sys.setenv(
    H2_PROJECT_DIR = project_dir,
    H2_GOVERNANCE_CONTROL = governance_control
  )

  run_log <- data.frame()

  for (notebook in pipeline_notebooks) {
    message("\nExecuting: ", notebook)
    started_at <- Sys.time()

    status <- system2(
      command = jupyter_command,
      args = c(
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        "--inplace",
        "--ExecutePreprocessor.kernel_name=ir",
        "--ExecutePreprocessor.timeout=-1",
        shQuote(file.path(notebook_dir, notebook))
      )
    )

    if (!identical(as.integer(status), 0L)) {
      stop(
        paste(
          "Pipeline stopped because execution failed for:",
          notebook
        )
      )
    }

    output_check <- check_fresh_outputs(
      expected_outputs[[notebook]],
      started_at
    )
    output_ok <- with(
      output_check,
      exists & nonempty & updated_this_run
    )

    if (any(!output_ok)) {
      print(output_check)
      stop(
        paste(
          "Missing, empty or stale output after",
          notebook
        )
      )
    }

    run_log <- rbind(
      run_log,
      data.frame(
        notebook = notebook,
        governance_control = governance_control,
        started_at_utc = format(
          started_at,
          tz = "UTC",
          usetz = TRUE
        ),
        finished_at_utc = format(
          Sys.time(),
          tz = "UTC",
          usetz = TRUE
        ),
        status = "succeeded",
        stringsAsFactors = FALSE
      )
    )

    utils::write.csv(
      run_log,
      file.path(
        project_dir,
        "pipeline_execution_log.csv"
      ),
      row.names = FALSE
    )
  }

  run_log
}

pipeline_started_at <- Sys.time()
pipeline_execution_log <- run_pipeline()
print(pipeline_execution_log, row.names = FALSE)


## Verify main and supplementary outputs


In [ ]:
main_output_check <- check_fresh_outputs(
  unname(main_outputs),
  pipeline_started_at
)
main_output_check <- cbind(
  figure = names(main_outputs),
  main_output_check
)
print(main_output_check, row.names = FALSE)

if (any(!with(main_output_check, exists & nonempty & updated_this_run))) {
  stop(
    "At least one main Figure 1–6 output was not regenerated successfully."
  )
}

utils::write.csv(
  main_output_check,
  file.path(project_dir, "main_output_check.csv"),
  row.names = FALSE
)

supplementary_output_check <- check_fresh_outputs(
  unname(supplementary_outputs),
  pipeline_started_at
)
supplementary_output_check <- cbind(
  figure = names(supplementary_outputs),
  supplementary_output_check
)
print(supplementary_output_check, row.names = FALSE)

if (
  any(
    !with(
      supplementary_output_check,
      exists & nonempty & updated_this_run
    )
  )
) {
  stop(
    "At least one S1–S7 output was not regenerated successfully."
  )
}

utils::write.csv(
  supplementary_output_check,
  file.path(project_dir, "supplementary_output_check.csv"),
  row.names = FALSE
)

message(
  "All notebooks completed with governance_control = '",
  governance_control,
  "'. Main Figures 1–6 and Supplementary Figures S1–S7 were regenerated."
)


## Reproducibility checks

The master runner verifies that every expected main and supplementary PDF is newly written during the run and that the required fitted-model outputs are present. The selected governance control is recorded in the pipeline execution log and in the fitted progression/failure and multinomial model objects. The competing-risk model is estimated once and reused by S3–S5.
